#### Reading posthoc CSV stats
Start 5/5/25. To gather all posthoc csvs exported, unite into one file
v2-5/31/26- add supp figure handling and addition into megafolder 
v3- 
- new null-band percentiles now flow through to the final table as combined Group 1/2 Null Band (5-95%) columns.
- Reorganized supplement panel assignments: added the current supp-CCG names to the panel map and removed stale duplicates.
- Fixed CSV-selection logic so each table contributes its  most-recent file, vs using one global latest date

#### Setup


In [ ]:
%pip install -r requirements.txt
from pathlib import Path
import os
import notebook_setup
info = notebook_setup.setup()

# Environment & Imports Setup
import matplotlib as matplotlib
%matplotlib inline 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from datetime import datetime
import itertools
import ast
##customs 
from helper_functions import *

## Preprocess/Gather Files

#### Read and gather all datasets 

In [ ]:
#Set/create save and load folder paths 
here = info["repo_root"]
print(f" Here: {here}")
results_location = Path(here).parents[0] / "results"
data_location = Path(here).parents[0] / "data"
print(f"Saving results in {results_location}. Data location is {data_location}")
csv_folder_most_recent = results_location/ f"analysis_CSV_output/" #folders that analysis output goes to
os.chdir(results_location)
print(os.getcwd())


In [ ]:
folders = []
with os.scandir(os.getcwd()) as dir_iter:
    for item in dir_iter:
        # print(item)
        if item.is_dir():
            folders.append(item)
folders

#### Sort folders in directory 

In [ ]:
#sort folders in dir by last edited
sorted_folders = sorted(folders, reverse = True, key = lambda entry: entry.stat().st_mtime)
sorted_folders[:10]

In [ ]:
sorted_csv_storage = [c for c in sorted_folders if 'analysis_CSV_output' in c.name]
print(sorted_csv_storage)
most_recent_csv_storage = sorted_csv_storage[0]
most_recent_csv_storage

#### get CSV folders in dir

In [ ]:
## navigate
os.chdir(most_recent_csv_storage)
folder = os.getcwd()
print(folder)
csv_files = [f for f in os.listdir(folder) if f.lower().endswith(".csv")]
print(f" {len(csv_files)} csv files found in folder")

In [ ]:
csv_files

#### find latest run of data 

In [ ]:
def nearest(items, pivot):
    return min(items, key=lambda x: abs(x - pivot)) 
    
def extract_date_str(csv_str, suffix = '.csv', split_delim = '_', get_post_split = -3):
    return csv_str.split(suffix)[0].split(split_delim)[get_post_split:]

In [ ]:
date_list = [" ".join(extract_date_str(x)) for x in csv_files ] #drop .csv, then get last 3 entries, but only if has any digit in str
dates_clean = [x for x in date_list if sum(i.isdigit() for i in x) > 5]
dates_clean[-10:]

In [ ]:
now = datetime.today()
print(now)

#### for each unique fig number, find the csvs with the min timedelta datetime, and collect

In [ ]:
def store_csv_name_by_fig_num(csv_files, max_fig_num = 8):
    ''' To- create dict where key = fig num and val = list of csv names with dates in title'''
    num_store = {f: list() for f in range(max_fig_num)}
    ## ## loop throuhg all CSVs
    for f in csv_files:
        fig_n = f.split("_")[0]
        if all((i.isdigit() for i in fig_n)):        # print(fig_n)
            num_store[int(fig_n)].append(f)
    return num_store

In [ ]:
num_store = store_csv_name_by_fig_num(csv_files)
num_store

#### create csv store


In [ ]:
results_location

In [ ]:
csv_store_folder = results_location / "figure_tables"
make_folder(csv_store_folder)

## Combine/Save Tables

#### test combination using figure 3 concat:

In [ ]:
#
def get_last_fig_csv_names(num_store:dict, fig_num:int, skip_flag = ['anova', 'cells active per trial'],**kwargs):
    ''' For each distinct table in the figure bucket (filename minus its trailing _DD_Mon_YYYY date),
    keep only that table's most-recent dated file. This avoids dropping supplements saved on a
    different date than other tables that share the same fig-number bucket (e.g. many 2_s_* files
    from different notebooks/runs). Returns (current_files, latest_overall_datetime).'''
    fig_storage = num_store[fig_num]
    valid_figs = [x for x in fig_storage if all([skip.lower() not in x.lower() for skip in skip_flag])] #skip_flag: list of str, verify all not present
    latest_by_base = {}  # base name (no date) -> (filename, datetime)
    for f in valid_figs:
        base = "_".join(f.split(".csv")[0].split("_")[:-3])  # strip trailing _DD_Mon_YYYY date tokens
        f_date = datetime.strptime(" ".join(extract_date_str(f)), '%d %b %Y')
        if base not in latest_by_base or f_date > latest_by_base[base][1]:
            latest_by_base[base] = (f, f_date)
    current_files = [fname for fname, _ in latest_by_base.values()]
    closest_time = max((dt for _, dt in latest_by_base.values()), default=datetime.today())
    print(f"fig {fig_num}: {len(current_files)} table(s) selected (latest per table)")
    return current_files, closest_time
    
#get list of dates in csvs for figure of interest, then find closest
def get_latest_csv_datetime(fig_storage:list,skip_flag:list = ["skip"], **kwargs):
    ''' To iterate over list of .csv filenames (with datetime tags embedderd) and find the closest embedded tag to the current time. 
    Skip_flag: list of strings to iterate through and not include in csv list if present'''
    now = datetime.today()
    valid_figs = [x for x in fig_storage if all([skip.lower() not in x.lower() for skip in skip_flag])] #skip_flag: list of str, verify all not present

    date_list_datetime = [datetime.strptime(" ".join(extract_date_str(x)),'%d %b %Y') for x in valid_figs]
    closest_time = min(date_list_datetime , key = lambda x: now- x )
    last_datetime_str = closest_time.strftime('%d_%b_%Y')#convert to str to match save formatting
    print(f"closest datetime: {closest_time}. last_datetime_str = {last_datetime_str}")
    return closest_time, last_datetime_str

def get_fig_csv_matching_datetime(fig_storage, closest_time=datetime.today(), skip_flag:list = ["skip"],):
    ''' To loop over input list of .csv filenames, and pick entries matching the previously found closest datetime'''
    current_files = list()
    #optional- filter list and ignore ones with certain content
    valid_figs = [x for x in fig_storage if all([skip.lower() not in x.lower() for skip in skip_flag])] #skip_flag: list of str, verify all not present
    # print(valid_figs)
    for f in valid_figs:## given closest datetime, iterate and extract 
        f_date = datetime.strptime(" ".join(extract_date_str(f)),'%d %b %Y')# #get string date and convert to datetime to compare against last known date entry
        if f_date == closest_time:
            print(f'matching date file: {f}')
            current_files.append(f)
    
    return current_files


In [ ]:
def concat_clean_csv_df(current_files, **kwargs):
    current_dfs = []
    for f in current_files:
        df = pd.read_csv(f)
        df['csv_date'] = " ".join(extract_date_str(os.path.basename(f)))  # e.g. '26 May 2026'
        df['csv_filename'] = os.path.basename(f)  # optional, for full traceability
        current_dfs.append(df)
    combined_fig_df = pd.concat(current_dfs)

    cols_to_drop = ['g_1_child_x', 'g_1_child_y', 'y_is_nonnan', 'y_is_2_elem', 'hue_is_x_axis',
                    'group_1_order_pos', 'group_2_order_pos', 'g_2_child_x', 'g_2_child_y',
                    'tick_text', 'tick_pos', 'hue_group_1_locs', 'hue_group_2_locs', 'Unnamed: 0',
                    'g1_num_loc', 'g2_num_loc', 'g1_cat_loc', 'g2_cat_loc', 'max_group_loc_val', 'plot_name']
    combined_fig_df.drop([c for c in cols_to_drop if c in combined_fig_df.columns], axis=1, inplace=True)
    return combined_fig_df


In [ ]:
get_latest_csv_datetime(num_store[1],skip_flag = ['anova', 'trial'])

In [ ]:
fig_num = 3
current_files, closest_time = get_last_fig_csv_names(num_store, fig_num)
combined_fig_df= concat_clean_csv_df(current_files)

combined_fig_df.group_1_n = combined_fig_df.group_1_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)
combined_fig_df.group_2_n = combined_fig_df.group_2_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)

#key value store for old: new col name
clean_col_name_dict = {'category_compared_within': "Group of posthoc comparison", 
                       'group_1': "Group 1",
                       'group_2': "Group 2", 
                       'group_1_n': "Group 1 N",
                       'group_2_n': "Group 2 N", 
                       'group_1_mean': "Group 1 Mean",
                       'group_1_sem': "Group 1 SEM",
                       'group_2_mean': "Group 2 Mean",
                       'group_2_sem':"Group 2 SEM",
                       'test_name': "Name of Statistical Test",
                       'stat_result': "Test Result",
                       'pvalue': "Test p-value",
                       'categorical_subgroup': "Alternate name- group compared within",
                       'numeric_var': "Variable Compared between Groups",
                       'hue_var': "Variable labeling post-hoc group",
                       'x_category_var': "Categorical variable of plot (x-axis)",
                       'date_tag': "Date of figure creation",
                       'fig_name': "filename of source table",
                       'fig_num': "Figure number"
                      }

combined_fig_df.rename(clean_col_name_dict, axis = 1,inplace = True)
combined_fig_df

In [ ]:
csv_name = f"fig_{fig_num}_results.csv"
combined_fig_df.to_csv( csv_store_folder/csv_name)

#### Concat posthoc statistics from figures 3-7:

In [ ]:
## as figs 1 and 2 don't use traditional posthoc test df outputting, skip that for now 
fig_start = 2
fig_end = 8
##
figs_to_concat = [k for k, v in num_store.items() if v]
print(f" Combining csvs with keys: {figs_to_concat}")
all_fig_tables = []
for fig_num in figs_to_concat:
    current_files, closest_time = get_last_fig_csv_names(num_store, fig_num)
    combined_fig_df= concat_clean_csv_df(current_files)
    combined_fig_df.group_1_n = combined_fig_df.group_1_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)
    combined_fig_df.group_2_n = combined_fig_df.group_2_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)
    #key value store for old: new col name
    clean_col_name_dict = {'category_compared_within': "Group of posthoc comparison", 
                           'group_1': "Group 1",
                           'group_2': "Group 2", 
                           'group_1_n': "Group 1 N",
                           'group_2_n': "Group 2 N", 
                           'group_1_mean': "Group 1 Mean",
                           'group_1_sem': "Group 1 SEM",
                           'group_2_mean': "Group 2 Mean",
                           'group_2_sem':"Group 2 SEM",
                           'test_name': "Name of Statistical Test",
                           'stat_result': "Test Result",
                           'pvalue': "Test p-value",
                           'categorical_subgroup': "Alternate name- group compared within",
                           'numeric_var': "Variable Compared between Groups",
                           'hue_var': "Variable labeling post-hoc group",
                           'x_category_var': "Categorical variable of plot (x-axis)",
                           'date_tag': "Date of figure creation",
                           'fig_name': "filename of source table",
                           'fig_num': "Figure number"
                          }
    combined_fig_df.rename(clean_col_name_dict, axis = 1,inplace = True)
    all_fig_tables.append(combined_fig_df)
    csv_name = f"fig_{fig_num}_concat_results.csv"
    combined_fig_df.to_csv( csv_store_folder/ csv_name)
full_fig_table = pd.concat(all_fig_tables)


In [ ]:
full_fig_table

In [ ]:
## save NON STANDARD tables to separate csv for review, then drop from main table
non_standard = full_fig_table[full_fig_table['filename of source table'].isna()]#.dropna(axis=1, how='all')
non_standard.to_csv(csv_store_folder / "non_standard_figure_tables.csv", index=False)
non_standard


In [ ]:
full_fig_table = full_fig_table.dropna(subset=['filename of source table']).dropna(axis=1, how='all')
full_fig_table


In [ ]:
unique_figs= sorted(full_fig_table['filename of source table'].unique())
# unique_figs

## create dict mapping key (filename) to value (corresponding figure panel)
map_panel_to_filename = {'Autoencoder Early_IA_Correct_v_Early_RS_Correct DB index by ensembles': "6G" ,
 'Autoencoder Early_IA_Correct_v_Late_IA DB index by ensembles': "7D",
 'Autoencoder Early_IA_Error_v_Early_RS_Error DB index by ensembles': "5H" ,
 'Autoencoder Late_IA_v_Early_RS_Correct DB index by ensembles': "7H" ,
 'CCG single class pred- Train Correct test Error- Early_IA_Correct ensemble': "4C" ,
 'CCG single class pred- Train Correct test Error- Early_RS_Error ensemble': "4F" ,
 'CCG single class pred- Train IA test RS - Early_RS_Correct ensemble': "3C" ,
 'Early_IA_Correct_v_Early_RS_Correct SVM accuracy by ensem': "6C" ,
 'Early_IA_Correct_v_Late_IA SVM accuracy by ensem': "7C" ,
 'Early_IA_Error_v_Early_RS_Error SVM accuracy by ensem': "5C" ,
 'Late_IA_v_Early_RS_Correct SVM accuracy by ensem': "7G" ,
 'time-dep decoding - Early RS Correct ens- Early_IA_Correct_v_Early_RS_Correct': "6D" ,
 'time-dep decoding - Early RS Error ens- Early_IA_Error_v_Early_RS_Error': "5D"
                        }
map_supp_panel_to_file = { 'Mean % of cells active in stage that are in stage ensemble':"S1I",
                           'supp_wtclnz_pointplot_Mean event rate by phase':"S1C",
                           'supp_Proportion frames active per trial': "S1B",
 # supp Autoencoder DB-index panels: land on Supplementary Figure 5
    'supp_Autoencoder Early_IA_Correct_v_Early_RS_Correct DB index by ensembles': "S5I" ,
 'supp_Autoencoder Early_IA_Error_v_Early_RS_Error DB index by ensembles': "S5G" ,
 # supp CCGP uniproportion-CCG prediction panels (current '+null' fig_names from SVM v5 cells 76-78): Supplementary Figure 4
 # NOTE: stale 'supp_CCG single class pred-' keys removed -- with per-table CSV selection they
 # would re-surface old-dated files and duplicate the S4C row.
 'supp_CCG+null single class pred- Train IA test RS - Early_RS_Correct ensemble': "S4C",
 'supp_CCG + null 1_class pred-Train_Correct_Test_Error- Early_RS_Error ensemble': "S4E",
 'supp_CCG + null 1_class pred-Train_Correct_Test_Error-Early_IA_Correct ensemble': "S4D"
                         }

legacy_map = {
    'null_CCGP- Train on IA trials, Test on RS trials_Early stage ens_ ': '3B',
    'null CCGP (early ens) train_correct_test_error': '4B',
    'null_CCG uni-class pred- Train IA test RS - Early_RS_Correct ensemble': '3C',
    'null_ CCG uni-class pred- Train Correct test Error- Early_IA_Correct ensemble': '4C',
    'null_CCG uni-class pred- Train Correct test Error- Early_RS_Error ensemble': '4F',
    'Supplement- VEH v CLNZ for Het & WT- ensemble proportion overlap': 'S2B',  
}


full_filename_panel_map = {**map_panel_to_filename, **map_supp_panel_to_file, **legacy_map}
full_filename_panel_map

#### prepare and save concat figure DF

In [ ]:
def clean_pvalue_string(x:float): 
    if x < 0.0001:
        if x == 0:
            cleaned =    r"<< 1 x 10^15"
        else:
            cleaned =    f'{x:.2e}'.replace("e", " x 10^")
    else:
        cleaned = f"{x:.4f}"
    return cleaned
import re
panel_re = re.compile(r'^(s_)?(\d+)_?([A-Z]+)_')
## v2 update: to handle combo panels like "5F-H" where multiple letters are present, and to add "S" prefix for supp figs

def extract_panel_id(name): 
    if pd.isna(name):
        return None
    m = panel_re.match(name)
    if not m:
        return None
    supp_prefix = "S" if m.group(1) else ""
    fig_num = m.group(2)
    letters = m.group(3)
    # single letter → "5C"; combo → "5F-H"
    panel = letters if len(letters) == 1 else f"{letters[0]}-{letters[-1]}"
    return f"{supp_prefix}{fig_num}{panel}"


In [ ]:
# fresh apply
full_fig_table["Figure Panel"] = full_fig_table['filename of source table'].apply(extract_panel_id)

# definitive view: each unique filename → what panel did it get
view = (full_fig_table.groupby('filename of source table')
        .agg(n_rows=('Figure Panel', 'size'),
             figure_panel=('Figure Panel', 'first')))
print(view)
print(f"\nTotal rows: {len(full_fig_table)}")
print(f"Rows with panel: {full_fig_table['Figure Panel'].notna().sum()}")


In [ ]:
# valid_panels = list(full_filename_panel_map.values())
valid_panels = sorted(set(map_panel_to_filename.values())
                     | set(map_supp_panel_to_file.values())
                     | set(legacy_map.values()))

print(f" Keeping rows with panel IDs: {valid_panels}")
# step 1: regex extraction for panel-prefixed filenames (5C_, 7_C_, s_4_C_, etc.)
full_fig_table["Figure Panel"] = full_fig_table['filename of source table'].apply(extract_panel_id)
# step 2: legacy-map fallback for filenames without panel prefix (null_*, supp_*, etc.)
full_fig_table["Figure Panel"] = full_fig_table["Figure Panel"].fillna(
    full_fig_table['filename of source table'].map(full_filename_panel_map)
)
# step 3: keep only rows with a mapped panel (preserves combo tags like 5F-H, 6E-G)
full_fig_table = full_fig_table[full_fig_table["Figure Panel"].notna()]
print(full_fig_table["Figure Panel"].value_counts())
full_fig_table
 
 
# # valid_panels = list(full_filename_panel_map.values())
# valid_panels = sorted(set(map_panel_to_filename.values())
#                      | set(map_supp_panel_to_file.values())
#                      | set(legacy_map.values()))

# print(f" Keeping rows with panel IDs: {valid_panels}")
# full_fig_table["Figure Panel"]= full_fig_table['filename of source table'].replace(full_filename_panel_map)
# full_fig_table = full_fig_table[full_fig_table["Figure Panel"].isin(valid_panels)]## make sure you only keep table with real panels of interest
# full_fig_table


In [ ]:

## TEST RELATED PREPROCESS
#Rename varaible names
variable_map = {'prop_active_frames': "% of frames with events", 
                 'mean_rate': "Normalized event rate",
                 'active_in_trial': '% of cells active per trial',
                 'value': '% of samples classified',
                'accuracy':'SVM Accuracy',
                'mean_acc': 'Time-dependent SVM Accuracy',
                'DB_index':'Davies-Bouldin Index of Latent Space activity',
               }
full_fig_table['Variable Compared between Groups'] = full_fig_table['Variable Compared between Groups'].map(variable_map)
#find cohen's d 
cohen_d_mask =  full_fig_table['Name of Statistical Test'] == 'robust_cohen_d'
full_fig_table.loc[cohen_d_mask, 'Test Result'] = full_fig_table.loc[cohen_d_mask, 'Test Result'].str.replace(" ", ",").apply(ast.literal_eval) #aplpy to cohen's d testsing
#clean/redo testing 
test_name_clean = {'robust_cohen_d': "Robust Cohen's d",
                   "permutation_test": "Permutation Test",
                   'MWU': "Mann-Whitney U",
                   'chi_squared':"Chi-Squared"}
full_fig_table['Name of Statistical Test'] = full_fig_table['Name of Statistical Test'].map(test_name_clean) #replace var name with rea lnames 
#for MWU/permutation tests, replace first space occurance with comma then literal eval
test_stat_spaceless_mask = (full_fig_table['Name of Statistical Test'] == 'Permutation Test') | (full_fig_table['Name of Statistical Test'] == 'Mann-Whitney U')
full_fig_table.loc[test_stat_spaceless_mask, 'Test Result']= full_fig_table.loc[test_stat_spaceless_mask, 'Test Result'].str.replace(" ", ",", n = 1).apply(ast.literal_eval)
#clean p-values

full_fig_table['Test Statistic Variable'] = full_fig_table['Name of Statistical Test'].map({"Robust Cohen's d": "Robust Cohen's d",
                                                                                            "Permutation Test": "Mean permutation group diff.",
                                                                                            "Mann-Whitney U Test": "U-statistic",
                                                                                            "Chi-Squared Test": "Chi-Squared"})
full_fig_table['Test Statistic Value'] = full_fig_table['Test Result'].apply(lambda x: x[0])
full_fig_table['Test p-value'] = full_fig_table['Test p-value'].apply(lambda x: clean_pvalue_string(x))
#bugfix- force timebin in comparison string to avoid excel results as dates 
time_dep_rows = full_fig_table['filename of source table'].str.contains("time-dep")
full_fig_table.loc[time_dep_rows,'Group of posthoc comparison']= "timebins: "+ full_fig_table.loc[time_dep_rows,'Group of posthoc comparison']
full_fig_table.loc[time_dep_rows,'Alternate name- group compared within']= "timebins: "+ full_fig_table.loc[time_dep_rows,'Alternate name- group compared within']
full_fig_table.tail()


In [ ]:
## update with supplementary table 2 final format 
#reorder then drop figs
new_col_order = ['Figure Panel', 'Group of posthoc comparison', 'Group 1', 'Group 2','Variable Compared between Groups',
                 'Group 1 N', 'Group 2 N', 'Group 1 Mean', 'Group 1 SEM', 'Group 2 Mean', 'Group 2 SEM',
                 'Name of Statistical Test', 'Test p-value','Test Statistic Variable','Test Statistic Value',
                 'Variable labeling post-hoc group',
                 'Categorical variable of plot (x-axis)', 
                 'filename of source table',
                 ]

## final supplement table map
col_rename_map = {'Variable Compared between Groups': 'Dependent Variable',
                  'Name of Statistical Test': 'Statistical Test',
                  'Group of posthoc comparison': 'Posthoc Comparison Group',
                  'Test p-value': 'p-value'}
## carry through null band (5th/95th pct) raw columns when present (CCGP figs 3B/4B)
null_band_raw_cols = ['group_1_null_low', 'group_1_null_high', 'group_2_null_low', 'group_2_null_high']
new_col_order = new_col_order + [c for c in null_band_raw_cols if c in full_fig_table.columns]
full_fig_table = full_fig_table.loc[:, new_col_order].rename(columns = col_rename_map)
## combine 'Group 1' / 'Group 2' name columns into a single 'Group 1 & 2' column (joined with ' vs. '), in place
full_fig_table.insert(full_fig_table.columns.get_loc('Group 1'), 'Group 1 & 2',
                      full_fig_table.apply(lambda x: f"{x['Group 1']}-{x['Group 2']}", axis=1))
full_fig_table = full_fig_table.drop(columns=['Group 1', 'Group 2'])
full_fig_table['shorthand test name'] = full_fig_table['Statistical Test'].map({"Robust Cohen's d": "d","Mann-Whitney U": "U"})
full_fig_table['Test stat., Name, Value']= full_fig_table.apply(lambda x: f'{x['shorthand test name']}={round(x["Test Statistic Value"],3)}', axis=1)
# full_fig_table['Grouping Variable ']= full_fig_table['Variable Compared between Groups']

full_fig_table

In [ ]:
## cleaning up posthoc comparisons
full_fig_table['Posthoc Comparison Group'] = full_fig_table['Posthoc Comparison Group'].str.replace("_", " ")
## time-dependent decoding clean up
time_dep_rows = full_fig_table['Posthoc Comparison Group'].str.contains("-")
full_fig_table.loc[time_dep_rows,'Posthoc Comparison Group'] = full_fig_table.loc[time_dep_rows,'Posthoc Comparison Group']+ " seconds from outcome"


In [ ]:
## v2 update- combine columns for space
#  create group 1, 2 N column
full_fig_table['N- Group 1 & 2 '] = full_fig_table.apply(lambda x: f"{x['Group 1 N']}, {x['Group 2 N']}", axis=1)
#  create +/- sem col
full_fig_table['Group 1 Mean +/- SEM'] = (full_fig_table['Group 1 Mean'].map('{:.3f}'.format)
    + ' +/- '  + full_fig_table['Group 1 SEM'].map('{:.3f}'.format))
full_fig_table['Group 2 Mean +/- SEM'] = (full_fig_table['Group 2 Mean'].map('{:.3f}'.format)
    + ' +/- ' + full_fig_table['Group 2 SEM'].map('{:.3f}'.format))
## null band (5th-95th percentile) combined into one string per group, like Mean +/- SEM
null_band_pairs = {'Group 1 Null Band (5-95%)': ('group_1_null_low', 'group_1_null_high'),
                   'Group 2 Null Band (5-95%)': ('group_2_null_low', 'group_2_null_high')}
for band_col, (lo_col, hi_col) in null_band_pairs.items():
    if lo_col in full_fig_table.columns and hi_col in full_fig_table.columns:
        full_fig_table[band_col] = full_fig_table.apply(
            lambda x, lo=lo_col, hi=hi_col: f"{x[lo]:.3f} - {x[hi]:.3f}" if pd.notna(x[lo]) and pd.notna(x[hi]) else "",
            axis=1)
full_fig_table.head()

#### Save full figure table after concats 

In [ ]:
#save fig table
cols_drop_in_save = ['Test Result', 
                     'Alternate name- group compared within', 
                     'comparison',
                     'Figure number',
                     'Date of figure creation',
                     'Variable labeling post-hoc group',
                     'Categorical variable of plot (x-axis)',
                     'shorthand test name', 
                     'Test Statistic Value',
                     'Test Statistic Variable',
                     'Group 1 N',
                    'Group 2 N',
                     'Group 1 Mean', 
                     'Group 1 SEM', 
                     'Group 2 Mean',
                    'Group 2 SEM',
                     'group_1_null_low', 'group_1_null_high',
                     'group_2_null_low', 'group_2_null_high'] ## drop any columns in this list that are still present in the table before saving, to avoid saving extraneous info

full_fig_table = full_fig_table.drop([c for c in cols_drop_in_save if c in full_fig_table.columns],axis = 1)
csv_name = f"all_figure_concat_results.csv"
full_fig_table.set_index("Figure Panel").to_csv( csv_store_folder/csv_name)
full_fig_table

In [ ]:
full_fig_table['Figure Panel'].value_counts()

In [ ]:
full_fig_table.iloc[:, -2:].value_counts()